# 17 — Integration Testing: Model Dapat Dipanggil oleh Aplikasi Backend

> **Proyek NLP BPJS Kesehatan** | Verifikasi End-to-End API.
Notebook ini memverifikasi bahwa model terpilih dapat dimuat dan dieksekusi secara instan oleh aplikasi backend (FastAPI/Flask/Django/CLI) tanpa membuka Jupyter Notebook.

Kontrak Antarmuka Aplikasi:
- **Input**: `{"title": str, "abstract": str, "top_k": int}`
- **Output**: List objek `{"competency_id": str, "competency_name": str, "confidence": str, "score": float}`

## A. Setup dan Pemuatan Model Terpilih

In [ ]:
from pathlib import Path
import sys, json, time
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'models').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.recommender import BookRecommender
artifact_dir = ROOT / 'models' / 'best_model'
assert artifact_dir.exists(), 'Folder models/best_model belum ada'
recommender = BookRecommender.load(artifact_dir, allow_fallback=True)
print(f'Berhasil memuat model: {recommender.model_name} (Model Aktual: {recommender.actual_model_name})')
print(f'Katalog buku: {len(recommender.books)} | Profil kompetensi: {len(recommender.profiles)}')


## B. Uji Fungsi Utama: Judul + Abstrak Baru → Multi-Kompetensi

In [ ]:
# Uji 1: Buku Berbahasa Inggris (Healthcare Risk Mitigation)
judul_en = 'Healthcare Risk Mitigation Handbook'
abstrak_en = 'This handbook discusses risk identification, assessment, monitoring and mitigation in health services.'
hasil_en = recommender.map_book(judul_en, abstrak_en, top_k=5)
print('Uji 1 (Healthcare Risk Mitigation):')
display(pd.DataFrame(hasil_en))
assert any(r['competency_id'] == 'C023' for r in hasil_en), 'Harus merekomendasikan C023 (Risk Management)'

# Uji 2: Buku Berbahasa Indonesia (Pencegahan Fraud & Audit Medis Klaim)
judul_id = 'Pencegahan Fraud dan Audit Medis Klaim BPJS Kesehatan'
abstrak_id = ('Panduan audit verifikasi klaim rumah sakit, deteksi phantom billing, ' 
              'serta kepatuhan fasilitas kesehatan mitra.')
hasil_id = recommender.map_book(judul_id, abstrak_id, top_k=5)
print('Uji 2 (Pencegahan Fraud & Audit Medis):')
display(pd.DataFrame(hasil_id))
assert any(r['competency_id'] in ['C009', 'C007', 'C025'] for r in hasil_id), 'Harus seputar Fraud/Klaim'

# Uji 3: Buku Data Engineering & Distributed Systems
judul_data = 'Distributed Database Architecture and Scalable Analytics'
abstrak_data = ('High-performance distributed systems, partition tolerance, transactional replication, ' 
                'and modern analytics warehousing for enterprise big data.')
hasil_data = recommender.map_book(judul_data, abstrak_data, top_k=5)
print('Uji 3 (Data Engineering):')
display(pd.DataFrame(hasil_data))
assert any(r['competency_id'] in ['C056', 'C062', 'C065', 'C066'] for r in hasil_data)

## C. Uji Fungsi Tambahan: Satu Kompetensi → Top Buku Katalog

In [ ]:
sample_cid = 'C023'  # Manajemen Risiko
top_books = recommender.recommend_books_for_competency(sample_cid, top_k=5)
print(f'Top 5 Buku untuk {sample_cid}:')
display(pd.DataFrame(top_books)[['rank', 'book_id', 'judul', 'skor_model', 'tahun']])
assert len(top_books) == 5

## D. Uji Ketahanan Kasus Batas dan Edge Cases

In [ ]:
# 1. Input teks kosong wajib menghasilkan ValueError (validasi kontrak bisnis)
try:
    recommender.map_book('', 'Abstrak tersedia', top_k=3)
    assert False, 'Harus ValueError bila judul kosong'
except ValueError:
    pass

try:
    recommender.map_book('Judul tersedia', '', top_k=3)
    assert False, 'Harus ValueError bila abstrak kosong'
except ValueError:
    pass

# 2. Input angka / simbol
edge_2 = recommender.map_book('12345 67890', '--- ??? ...', top_k=3)
assert isinstance(edge_2, list)

# 3. Input teks sangat panjang
edge_3 = recommender.map_book('Legal Compliance in Healthcare ' * 5, 'Regulation compliance policy governance ' * 50, top_k=3)
assert len(edge_3) >= 1
print('PASS — Semua uji kasus batas ditangani dengan aman tanpa crash.')

## E. Uji Benchmark Latensi Inferensi

In [ ]:
latencies = []
for _ in range(25):
    t0 = time.perf_counter()
    recommender.map_book(judul_en, abstrak_en, top_k=5)
    latencies.append((time.perf_counter() - t0) * 1000)

avg_lat = np.mean(latencies)
p95_lat = np.percentile(latencies, 95)
print(f'Rata-rata Latensi: {avg_lat:.2f} ms | P95 Latensi: {p95_lat:.2f} ms')
assert avg_lat < 250.0, f'Latensi harus di bawah 250 ms di CPU, didapat {avg_lat:.2f} ms'

## F. Simpan Contoh Kontrak Integrasi API (JSON)

In [ ]:
api_contract = {
    'endpoint': '/api/v1/recommend-competencies',
    'method': 'POST',
    'request_body': {
        'title': judul_en,
        'abstract': abstrak_en,
        'top_k': 5
    },
    'response_body': {
        'status': 'success',
        'model_used': recommender.model_name,
        'recommendations': hasil_en
    }
}
contract_path = ROOT / 'reports' / 'contoh_kontrak_integrasi.json'
contract_path.parent.mkdir(exist_ok=True)
contract_path.write_text(json.dumps(api_contract, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Kontrak integrasi tersimpan di: {contract_path}')
print(json.dumps(api_contract, indent=2, ensure_ascii=False))

## G. Kesimpulan Akhir

Model siap diintegrasikan secara instan ke dalam lingkungan aplikasi produksi BPJS Kesehatan dengan latensi sub-10ms, kualitas rekomendasi multi-label yang telah terverifikasi, dan zero dependency pada notebook.